# Starfysh sample-integration tutorial in scviva-tools

This notebook adapts the upstream Starfysh integration tutorial to the current scviva ecosystem.

The upstream notebook performs joint sample integration with optional PoE histology. That full path is not implemented yet in scviva-tools. The active cells below show the supported Phase 2A workflow: run expression-only Starfysh per sample, store proportions and latent representations, and concatenate results for downstream comparison.


In [ ]:
!pip install --quiet scviva-tools


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import torch

import scviva
from scviva.external import Starfysh

scviva.settings.seed = 0
torch.manual_seed(0)


In [ ]:
def _as_dense(x):
    if hasattr(x, "toarray"):
        return x.toarray()
    return np.asarray(x)


def compute_signature_scores(adata, gene_signatures):
    """Compute simple per-spot signature priors from marker-gene columns.

    The current scviva Starfysh wrapper expects one prior score per spot and
    cell type. Upstream Starfysh notebooks often start from marker-gene lists;
    this helper turns those marker lists into normalized spot-level priors.
    """
    signatures = gene_signatures.copy()
    if "Unnamed: 0" in signatures.columns:
        signatures = signatures.drop(columns=["Unnamed: 0"])

    scores = pd.DataFrame(index=adata.obs_names)
    for cell_type in signatures.columns:
        markers = signatures[cell_type].dropna().astype(str)
        markers = [gene for gene in markers if gene in adata.var_names]
        if len(markers) == 0:
            scores[cell_type] = 0.0
            continue
        values = _as_dense(adata[:, markers].layers.get("counts", adata[:, markers].X))
        scores[cell_type] = values.mean(axis=1)

    scores = scores.clip(lower=0)
    row_sums = scores.sum(axis=1).replace(0, np.nan)
    return scores.div(row_sums, axis=0).fillna(1.0 / scores.shape[1])


## Configure samples and signatures

Update `data_path`, `meta_info`, and `sig_file_name` to match your local copy of the upstream Starfysh integration data.


In [ ]:
data_path = Path("data")
sig_file_name = "bc_signatures_version_1013.csv"
meta_info = pd.DataFrame(
    [
        ["P1A_ER", "P1_ER", "ER"],
        ["CID44971", "CID44971_TNBC", "TNBC"],
    ],
    columns=["sample", "patient", "tissue_type"],
)

gene_sig = pd.read_csv(data_path / sig_file_name)
meta_info


## Run current scviva Starfysh per sample

This is not the upstream joint integration model. It is the currently supported scviva-compatible bridge: each sample is deconvolved with the same signature table, and the resulting latent/proportion arrays are stored in AnnData for comparison.


> **Note: raw counts required.** The Starfysh model expects raw integer count data in the registered layer (default: `adata.X`). Do **not** log-normalize or scale before calling `setup_anndata` — the model applies its own library-size normalization internally. Normalizing beforehand will silently degrade training.

In [ ]:
sample_adatas = []
proportion_tables = []

device = "cuda" if torch.cuda.is_available() else "cpu"
for row in meta_info.itertuples(index=False):
    sample_id = row.sample
    candidate_h5ads = [
        data_path / sample_id / "st.h5ad",
        data_path / sample_id / "adata.h5ad",
        data_path / f"{sample_id}.h5ad",
    ]
    for adata_path in candidate_h5ads:
        if adata_path.exists():
            break
    else:
        raise FileNotFoundError(f"Could not find AnnData for sample {sample_id}.")

    adata = sc.read_h5ad(adata_path)
    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()
    if "spatial" not in adata.obsm:
        coords_path = data_path / sample_id / "spot_list.csv"
        coords = pd.read_csv(coords_path, index_col=0).reindex(adata.obs_names)
        adata.obsm["spatial"] = coords.iloc[:, :2].to_numpy(dtype=np.float32)

    adata.obs["sample"] = sample_id
    adata.obs["patient"] = row.patient
    adata.obs["tissue_type"] = row.tissue_type

    signature_scores = compute_signature_scores(adata, gene_sig)
    Starfysh.setup_anndata(adata, layer="counts", spatial_key="spatial")
    model = Starfysh(adata, signature_scores=signature_scores, n_latent=10, n_hidden=128)
    model.train(max_epochs=100, batch_size=128, lr=1e-3, device=device, prog_bar=True)

    outputs = model.get_model_outputs(batch_size=128, store=True)
    proportions = pd.DataFrame(
        outputs["qc_m"],
        index=adata.obs_names,
        columns=gene_sig.columns,
    )
    latent = outputs["qz_m"]

    proportion_tables.append(proportions)
    sample_adatas.append(adata)

adata_integrated = sc.concat(sample_adatas, keys=list(meta_info["sample"]))
if "sample" not in adata_integrated.obs.columns:
    adata_integrated.obs["sample"] = adata_integrated.obs_names.str.split("/").str[0]
proportions_integrated = pd.concat(proportion_tables, axis=0)
adata_integrated


In [ ]:
sc.pp.neighbors(adata_integrated, use_rep="X_starfysh")
sc.tl.umap(adata_integrated)
sc.pl.umap(adata_integrated, color=["sample", "patient", "tissue_type"], frameon=False)


## Deferred upstream joint-integration and PoE sections


In [ ]:
# TODO: Phase 4 — StarfyshPoEModule and histology patches
# Upstream reference only:
#   model = sf_model.AVAE_PoE(...)
#
# TODO: Phase 6 — integrated VisiumArguments and plotting helpers
# Upstream reference only:
#   integrated_args = utils_integrate.VisiumArguments_integrate(...)
#   model, loss = utils_integrate.run_starfysh(integrated_args, poe=poe_on, ...)
#   adata_integrate_starfysh = sf_model.model_eval_integrate(...)
